# FAOSTAT — farm production and producer prices

The UN Food and Agriculture Organization publishes free statistics for 245+
countries. CeyNex uses two things from it:

- **Producer prices** — what farmers were actually paid, in USD per tonne.
- **Crop production volumes** — how much was grown.

Connector: `ceynex/data/connectors/faostat.py` (owner: M1 Dinapura).
It reads CSVs exported from FAOSTAT's own site — it does **not** call an API.

In [1]:
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd

import _common as cx

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Is the data staged on this machine?

The agriculture connectors do not use a fixed path. `pipeline._agriculture_raw_dir`
tries three locations in order, and the first that exists wins:

1. `$CEYNEX_AGRICULTURE_RAW_DIR` — an explicit override
2. `data/raw/` **next to** the repo checkout
3. `ceynex-core/data/raw/`

Inside that, FAOSTAT reads every `*.csv` in the newest `YYYY-MM-DD` snapshot
directory.

In [2]:
AGRI = cx.agriculture_raw_dir()
print("agriculture raw dir:", AGRI or "none found")

FAOSTAT_DIR = AGRI / "faostat" if AGRI else None
csvs = sorted(cx.latest_snapshot(FAOSTAT_DIR).glob("*.csv")) if FAOSTAT_DIR and FAOSTAT_DIR.is_dir() else []

if csvs:
    print(f"\n{len(csvs)} CSV file(s) staged:")
    for path in csvs:
        print(" -", path.name)
else:
    print("\nFAOSTAT")
    print("  staged: NO — the analysis cells below will skip.")
    print(f"  expected at: {FAOSTAT_DIR}/<YYYY-MM-DD>/*.csv" if FAOSTAT_DIR else "  expected at: no raw dir")
    print("  how to get it: export the Sri Lanka producer-price and production")
    print("                 bulk CSVs from fao.org/faostat and drop them in.")

agriculture raw dir: /ml/CeyNex/ceynex-core/data/raw

FAOSTAT
  staged: NO — the analysis cells below will skip.
  expected at: /ml/CeyNex/ceynex-core/data/raw/faostat/<YYYY-MM-DD>/*.csv
  how to get it: export the Sri Lanka producer-price and production
                 bulk CSVs from fao.org/faostat and drop them in.


## 2. What the connector keeps

FAOSTAT's export is wide — every country, every crop, every measure. The
connector narrows it hard before anything is written:

| Filter | Value |
|---|---|
| `Area` | `Sri Lanka` |
| `Area Code (M49)` | `144` |
| `Element` | `Producer Price (USD/tonne)` — exact match |
| `Months` | `Annual value` |
| `Item` | the four crops below |

| FAOSTAT item | CeyNex item | HS code |
|---|---|---|
| Tea leaves | `tea` | 0902 |
| Cinnamon and cinnamon-tree flowers, raw | `cinnamon` | 0906 |
| Natural rubber in primary forms | `rubber` | 4001 |
| Coconuts, in shell | `coconut` | 0801 |

**The file is read with `encoding="latin1"`, not UTF-8.** FAOSTAT's bulk CSVs
contain Latin-1 bytes in country and item names; reading them as UTF-8 raises.

## 3. Two things this source deliberately throws away

Both are worth understanding, because both look like data loss until you know
why.

**Production volumes never reach `fact_trade`.** The frozen contract has no
`production_volume` column. The obvious workaround — writing production into
`export_volume` — would be wrong: how much tea Sri Lanka *grew* is not how much
it *exported*. Production stays staged, and the agriculture agent reads it from
staging rather than from `fact_trade`.

**Local-currency prices are dropped.** FAOSTAT reports the same item and year
twice, once in USD/tonne and once in local currency. `fact_trade` has no
currency field in its identity key, so both rows would have the same identity
and one would silently overwrite the other. Only the USD series is mapped.

The unit stays `USD/tonne` at this stage; `DataCleaner` normalises it to
`USD/kg` later, so the notebook below reports tonnes.

## 4. Load

In [3]:
raw = None
if csvs:
    frames = []
    for path in csvs:
        frame = pd.read_csv(path, encoding="latin1")
        frame["raw_file"] = path.name
        frames.append(frame)
    raw = pd.concat(frames, ignore_index=True)
    print(f"{len(raw):,} raw rows from {len(csvs)} file(s)")
    print("columns:", list(raw.columns))
    display(raw.head())
else:
    print("skipped — no FAOSTAT CSVs staged")

skipped — no FAOSTAT CSVs staged


## 5. Narrow to what the connector would write

This repeats `to_fact_trade`'s filter so you can see how many of the raw rows
actually survive it.

In [4]:
ITEMS = {
    "Tea leaves": ("tea", "0902"),
    "Cinnamon and cinnamon-tree flowers, raw": ("cinnamon", "0906"),
    "Natural rubber in primary forms": ("rubber", "4001"),
    "Coconuts, in shell": ("coconut", "0801"),
}

prices = None
if raw is not None:
    annual = (
        raw["Months"].astype("string").str.strip().eq("Annual value")
        if "Months" in raw.columns
        else pd.Series(True, index=raw.index)
    )
    prices = raw.loc[
        raw["Area"].astype("string").str.strip().eq("Sri Lanka")
        & pd.to_numeric(raw["Area Code (M49)"], errors="coerce").eq(144)
        & raw["Item"].isin(ITEMS)
        & raw["Element"].astype("string").str.fullmatch(r"Producer Price \(USD/tonne\)")
        & annual
    ].copy()
    prices["item"] = prices["Item"].map(lambda i: ITEMS[str(i)][0])
    prices["year"] = pd.to_numeric(prices["Year"], errors="coerce").astype("Int64")
    prices["price_usd_per_tonne"] = pd.to_numeric(prices["Value"], errors="coerce")
    prices = prices.dropna(subset=["price_usd_per_tonne"])
    print(f"{len(raw):,} raw rows -> {len(prices):,} rows that reach fact_trade")
    print("\nrows per item:")
    print(prices["item"].value_counts().to_string())
else:
    print("skipped — no data")

skipped — no data


## 6. Producer price over time

In [5]:
if prices is not None and not prices.empty:
    wide = prices.pivot_table(index="year", columns="item", values="price_usd_per_tonne", aggfunc="mean")
    ax = wide.plot(marker="o", title="FAOSTAT producer price, Sri Lanka (USD per tonne)")
    ax.set_ylabel("USD / tonne")
    ax.set_xlabel("Year")
    plt.tight_layout()
    plt.show()
    display(wide.tail(10).round(0))
else:
    print("skipped — no data")

skipped — no data


## 7. Coverage check

Producer-price series are often patchy for the most recent years — FAO revises
them late. This lists the first and last year present per crop, and any gaps in
between, so a forecast is never fitted across a hole without someone noticing.

In [6]:
if prices is not None and not prices.empty:
    for item, group in prices.groupby("item"):
        years = sorted(int(y) for y in group["year"].dropna().unique())
        gaps = sorted(set(range(years[0], years[-1] + 1)) - set(years))
        print(f"{item:10} {years[0]}-{years[-1]}  ({len(years)} obs)  gaps: {gaps or 'none'}")
else:
    print("skipped — no data")

skipped — no data
